### **Exponential CURVE FIT function algorithm for the mean value of chromosome_to_chromosome distance** 
This script fits an exponential curve to the **mean** of each cell type of the *C. elegans* embryo in batch process. This curve uses the exponential curve fitting function where $y = a * (1 - e^{-x/b})$. The goal for fitting a mathematical function to the mean value is to determine the *Final chromosome-to-chromosome length* and *Segregation speed* at the required time point.

#### `INPUT FILES` 
The csv files of each cell type containing the chromosome-to-chromosome distance data of different experiments (n-value). 
The table below shows the table structure of each .csv file. The headers of each table should show the time series measurement value from each observation which starts with prefix, **Exp**, and other statistical values from the observations.

| Exp00 | Exp01 | Exp02 | Exp03 | Exp04 | Exp05 | mean | std | n | SE | time |
| :-----: | :-----: | :-----: | :-----: | :-----: | :-----: | :-----: | :-----: | :-----:| :-----: | :-----: |
| `num` | `num` | `num` | `num` | `num` | `num` | `num` | `num` | `num`| `num` | `num` |
| `...` | `...` | `...` | `...` | `...` | `...` | `...` | `...` | `...`| `...` | `...` |


#### `OUTPUT FILES` 
* **FIRST_GROUP_OUTPUT_FILES**: The *.png* files of the fitted plot of each cell type. 

* **SECOND_GROUP_OUTPUT_FILE**: A *.csv* file containing the **Final chromosomes length (µm)** and the **Segregation speed (µm/minute)** of each of the cell type. 

* **THIRD_GROUP_OUTPUT_FILE**: An *.csv* file that compiles the spatiotemporal mean values resulting from the curve-fitting process for each input file.

In [7]:
# library packages
import os
import warnings
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from numpy import exp, linspace, random, arange
from scipy.optimize import curve_fit, least_squares

In [8]:
# input folder
folder = r'D:\data\Analysis Data\python_analysis\input'
fileTable = os.listdir(folder)

# output folder
save_files = r'D:\data\Analysis Data\python_analysis\output'

In [9]:
# create a new DataFrame to append the new generated table 
fit_Result = pd.DataFrame()
mean_fit_values = pd.DataFrame()
time_fit_values = pd.DataFrame()

# read out individual files and compute for different operations for each file
for file in os.scandir(folder):
    df = pd.read_csv(file)
    
    # create a new DataFrame 
    Exp_Column = df.loc[:, df.columns.str.startswith('Exp')]
    mean_column = df.loc[:, df.columns.str.startswith('mean')]
    time_column = df.loc[:, df.columns.str.startswith('time')]
    
    df_new = [Exp_Column, mean_column, time_column]
    df_Table = pd.concat(df_new, axis=1)
    
    '''
    drop all the rows were n-value is less than 3 the mean and the time 
    columns are included, ie, the number of rows to be computed should be >=3
    '''
    newTable = df_Table.dropna(thresh=3) 
    
    # Define the exponential function to fit in the data
    '''
    a = initial amplitude of the function
    b = time constant (the time taken for the function to reach approximately 63.2% of its maximum value, 
    i.e., 1-1/e as x approaches infinity)
    '''
    def exponential(x, a, b):
        return a * (1 - np.exp(-x / b))
    
    # ignore warning
    warnings.simplefilter(action="ignore", category=FutureWarning)
    
    # iterate through the desired columns header
    cells= [columnname for columnname in newTable if columnname.startswith('Exp')]

    # define x-values and y-values 
    x = newTable['time']
    y_mean = newTable['mean']

    # compute for the initial guess
    initial_guess = [10, 50]

    # summarize the parameters
    popt, pcov = curve_fit(exponential, x, y_mean, initial_guess, maxfev=10000)

    ''' 
    Define a sequence of inputs between the smallest and largest known inputs and define the fit. 
    Let the maximum input assume the maximum values of x. 
    '''
    x_fit = np.arange(0, x.max()+0.1, 0.1, dtype=None)
    y_fit = exponential(x_fit, *popt)
    
    plt.figure(figsize=(5,6))
    
    # plot the primary data
    for i_columns in cells: 
        ax1 = sns.scatterplot(data=newTable, x=x, y=i_columns, color='grey', alpha=0.2) 
    
    # add the desired features on the plot
    ax1.set_xlabel('time (sec)', fontsize= 20)
    ax1.set_ylabel('distance (μm)', fontsize= 20)
    plot_title = (file.name).split('.')[0]
    ax1.axes.set_title(plot_title, fontsize= 20, fontweight='bold')  
    ax1.set(ylim=(0, 10), xlim=(0, 200))
    
    # plot the mean on the primary plot
    sns.scatterplot(data=newTable, x=x, y=y_mean, color='brown', alpha=0.6, ax=ax1)
    
    # plot the fit on the primary plot
    sns.lineplot(x=x_fit, y=y_fit, alpha = 1, color='green', ax=ax1)
    
    # plot_files = os.path.join(save_files)
    plotfile = (file.name).split('.')[0] + '.png'
    plt.savefig(os.path.join(save_files, plotfile), dpi=300)
    
    # assign variables to the fit parameter output
    amp = popt[0]
    T = popt[1]
    
    # determine the slope == segregation speed
    segregation_speed = (amp/T)*60 # convert to µm/min by multiplying by 60
    
    # determine the maximum segregation length == final chromosome_to_chromosome distance
    final_C_C_length = amp
    
    # create a table for the speed of segregation and the maximum distance
    parameters = {'Final chromosomes length (µm)': final_C_C_length, 
                  'Segregation speed (µm/min)': segregation_speed}
    parameters_df = pd.DataFrame.from_dict(parameters, orient='index', columns=[file.name.split('.')[0]])
    fit_Result = pd.concat([fit_Result, parameters_df], axis=1)
    
    ''' 
    create a new DataFrame for the mean fro each curve-fitting
    '''
    fit_values = y_fit
    mean_fit = {file.name.split('.')[0]: fit_values}
    mean_fit_df = pd.DataFrame.from_dict(mean_fit)
    mean_fit_values = pd.concat([mean_fit_values, mean_fit_df], axis=1)

# close all open windows
plt.close('all')

# fit time values
time_fit = {'time': x_fit}
time_fit_df = pd.DataFrame.from_dict(time_fit)
time_fit_values = pd.concat([time_fit_values, time_fit_df], axis=1) 

# new DataFrame for the fit time and mean values
fit_Table = pd.concat([time_fit_values, mean_fit_values], axis=1)

# transpose the table
fit_Result_transpose = (fit_Result).T

# assign header to the index column
fit_Result_transpose.index.names = ['Cells']

# save the table, fit_Result, to a csv file
fit_Result_transpose.to_csv(os.path.join(save_files, 'Fit_Result_chromosome_chromosome.csv'))
fit_Table.to_csv(os.path.join(save_files, 'Fit_Result_chromosome_chromosome_mean_fit_values.csv'), 
                 index=False, encoding='utf-8')

C:\Users\thewo\AppData\Local\Temp\ipykernel_30496\1972656815.py:56: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(5,6))
